In [1]:
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import math


def save_graph(G, n, a):
	# nx.write_graphml(G, f"./graphs/G_n={n}_a={a}.graphml")
	nx.write_graph6(G, f"./graphs/G_n={n}_a={a}.g6")

def save_graphs(graph_list):
	for G, n, a in graph_list:
		save_graph(G, n, a)

def display_graphs(graphs, figsize=(15, 15), show_titles=True, force_row=False, with_labels=False):
	def axes_division(x):
		factor1 = 1
		for i in range(1, int(math.sqrt(x)) + 1):
			if x % i == 0:
				factor1 = i
		return int(factor1), int(x / factor1)

	# layout override
	if force_row:
		rows, cols = 1, len(graphs)
	else:
		rows, cols = axes_division(len(graphs))

	fig, axes = plt.subplots(rows, cols, figsize=figsize)

	# ensure axes is flat array
	axes = np.array(axes).flatten()

	for (G, n, a), ax in zip(graphs, axes):
		pos = nx.spring_layout(G, seed=100)
		nx.draw(
			G,
			pos,
			ax=ax,
			with_labels=with_labels,
			node_size=20,
			font_size=10,
			node_color=['red' if G.nodes[v].get("infected") else 'lightgray'
						for v in G],
		)

		if show_titles:
			ax.set_title(f"Graph C({n}, {a})")
		else:
			ax.set_title("")

		ax.set_aspect("equal")

	plt.tight_layout()
	plt.show()

def display_graph(G, figsize=(6, 6), show_title=True, with_labels=False, pos=None):
	fig, ax = plt.subplots(figsize=figsize)

	if pos == None:
		pos = nx.spring_layout(G, seed=100)

	nx.draw(
		G,
		pos,
		ax=ax,
		with_labels=with_labels,
		node_size=20,
		font_size=10,
		node_color=[
			'red' if G.nodes[v].get("infected") else 'lightgray'
			for v in G
		],
	)

	if show_title:
		ax.set_title(f"Graph C({len(G.nodes)})")
	else:
		ax.set_title("")

	ax.set_aspect("equal")
	plt.tight_layout()
	plt.show()

def generate_graphs(n):
	n_half = int(n/2)
	graphs = []
	
	def add_candidate_graph(H, a):
		nonlocal graphs
		for (G_i, _, _) in graphs:
			if nx.is_isomorphic(G_i, H):
				return
		graphs.append((H, n, a))
	
	for a in range(n_half):
		gcd = np.gcd(a, n_half)
		condition = (1 <= a < n_half and gcd == 1)
		
		if condition:
			g = nx.circulant_graph(n, [a, -a, n_half])
			add_candidate_graph(g, a)
			# print(f"a={a} n={n}")
	
	print(f"number of graphs for n={n} :", len(graphs))
	return graphs

def is_infectable(G, node):
	node_N = list(nx.all_neighbors(G, node))
	if len(node_N) < 2:
		return False
	
	infected_N_counter = sum(1 for i_neighbor in node_N 
							 if G.nodes[i_neighbor].get("infected"))
	return infected_N_counter >= 2

def fully_endemic(G_):
	G = G_.copy()
	max_itter_number = len(G.nodes)
	infected_nodes = [n for n in G.nodes if G.nodes[n].get("infected")]
	healthy_nodes = [n for n in G.nodes if not G.nodes[n].get("infected")]

	itter_number = 0
	while healthy_nodes:
		# find all new infections in THIS wave
		newly_infected = [node for node in healthy_nodes if is_infectable(G, node)]

		if not newly_infected:
			break

		# infect all of them at once
		for node in newly_infected:
			G.nodes[node]["infected"] = True

		# update tracking sets
		for node in newly_infected:
			infected_nodes.append(node)
			healthy_nodes.remove(node)

		itter_number += 1
		if itter_number > max_itter_number:
			break

	return (G, infected_nodes, healthy_nodes)

def infection_spread(G, n, a):
	graph_stages = [G.copy()]
	infected_nodes = [u for u in G.nodes if G.nodes[u].get("infected")]
	healthy_nodes = [u for u in G.nodes if not G.nodes[u].get("infected")]

	itter_number = 0
	max_iters = len(G.nodes)

	while healthy_nodes:
		# determine whole wave
		newly_infected = [node for node in healthy_nodes if is_infectable(G, node)]

		if not newly_infected:
			break

		# infect all of them simultaneously
		for node in newly_infected:
			G.nodes[node]["infected"] = True

		# update node lists
		for node in newly_infected:
			infected_nodes.append(node)
			healthy_nodes.remove(node)

		# add stage AFTER wave completes
		graph_stages.append(G.copy())

		itter_number += 1
		if itter_number > max_iters:
			break

	return [(H, n, a) for H in graph_stages], infected_nodes, healthy_nodes

def show_graph_infection(H):
	print("C=", end="")
	for j in H.nodes:
		if H.nodes[j].get("infected"):
			print(1, end="")
		else:
			print(0, end="")
	print()

def infection_graph_permuation(vertex_list, infection_number, z=0):
	# Base case: no more infections to place
	if infection_number == 0:
		yield vertex_list
		return

	# Try placing the next infection at any allowed index
	for i in range(z, len(vertex_list)):
		if vertex_list[i] == 0:
			new_list = vertex_list[:]  # copy
			new_list[i] = 1
			yield from infection_graph_permuation(new_list, infection_number - 1, i + 1)

def infect_graph(G, infection_mask):
	H = G.copy()
	for i, e in enumerate(infection_mask):
		H.nodes[i]["infected"] = bool(e)
	return H

def generate_graphs2(n):
	n_half = int(n/2)
	graphs = []
	
	def add_candidate_graph(H, a):
		nonlocal graphs
		for (G_i, n_i, a_i) in graphs:
			if nx.is_isomorphic(G_i, H):
				print(f"n={n_i} a={a_i} is isomorpic")
				return
		graphs.append((H, n, a))
	
	for a in range(n_half):
		gcd = np.gcd(a, n_half)
		condition = (1 <= a < n_half and gcd == 1)
		
		if condition:
			g = nx.circulant_graph(n, [a, -a, n_half])
			add_candidate_graph(g, a)
	
	print(f"number of graphs for n={n} :", len(graphs))
	return graphs


import math

def prism_layout(G):
    n = len(G.nodes)
    n_half = n // 2

    pos = {}

    r_inner = 1.0
    r_outer = 2.0

    for i in range(n_half):
        angle = 2 * math.pi * i / n_half

        # inner (top) cycle
        pos[i] = (
            r_inner * math.cos(angle),
            r_inner * math.sin(angle)
        )

        # outer (bottom) cycle
        pos[i + n_half] = (
            r_outer * math.cos(angle),
            r_outer * math.sin(angle)
        )

    return pos
